In [12]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML  # Importação necessária para o Colab

# ==========================================
# 1. Parâmetros do Circuito (Rg = 0.2 ohms)
# ==========================================
f = 60.0
omega = 2 * np.pi * f
T = 1.0 / f

V_peak = 900.0

R_g = 0.2  # Resistência do gerador
L_g = 15e-3  # Indutância do gerador
X_g = omega * L_g

R_L = 25.0  # Resistência da carga

# Impedâncias e Fases
Z1_mag = np.sqrt((R_g + R_L)**2 + X_g**2)
phi1 = np.arctan2(X_g, R_g + R_L)

Z2_mag = np.sqrt(R_g**2 + X_g**2)
phi2 = np.arctan2(X_g, R_g)

# Constante de tempo do transiente
tau = L_g / R_g

theta = 0.0

# Vetores de tempo
t_pre = np.linspace(-2*T, 0, 800, endpoint=False)
t_pos = np.linspace(0, 10*T, 2400)

# Corrente antes do curto
i_pre = (V_peak / Z1_mag) * np.sin(omega * t_pre + theta - phi1)
i_0 = (V_peak / Z1_mag) * np.sin(theta - phi1)

# Corrente depois do curto
i_cc_ac = (V_peak / Z2_mag) * np.sin(omega * t_pos + theta - phi2)
A = i_0 - (V_peak / Z2_mag) * np.sin(theta - phi2)
i_dc = A * np.exp(-t_pos / tau)
i_pos = i_cc_ac + i_dc

# Concatenando
t_total = np.concatenate((t_pre, t_pos))
i_total = np.concatenate((i_pre, i_pos))

# Criando o vetor total da componente CC (0 antes do curto, i_dc depois)
i_dc_pre = np.zeros_like(t_pre)
i_dc_total = np.concatenate((i_dc_pre, i_dc))

# ==========================================
# 2. Configuração da Animação para o Colab
# ==========================================
fig, ax = plt.subplots(figsize=(10, 5)) # Tamanho ligeiramente ajustado para caber bem na tela do Colab

# Limites do gráfico
ax.set_xlim([-2*T*1000, 10*T*1000])
y_max = np.max(np.abs(i_total)) * 1.1
ax.set_ylim([-y_max, y_max])

# Elementos estáticos do gráfico
ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Curto-Circuito (t=0)')
ax.set_title('Oscilografia Animada: Corrente de Curto-Circuito', fontsize=14, fontweight='bold')
ax.set_xlabel('Tempo (ms)', fontsize=12)
ax.set_ylabel('Corrente (A)', fontsize=12)
ax.grid(True, which='both', linestyle='-', linewidth=0.5, color='gray', alpha=0.7)
ax.minorticks_on()
ax.grid(True, which='minor', linestyle=':', linewidth=0.5, color='gray', alpha=0.5)

# As linhas que serão desenhadas quadro a quadro
linha_total, = ax.plot([], [], color='blue', linewidth=2, label='Corrente Total')
linha_cc, = ax.plot([], [], color='orange', linestyle='--', linewidth=2, label='Componente CC')

# Adiciona a legenda após criar as linhas para que elas apareçam corretamente
ax.legend(loc='upper right')

def init():
    linha_total.set_data([], [])
    linha_cc.set_data([], [])
    return linha_total, linha_cc

passo = 30 # Aumentei um pouquinho o passo para gerar o vídeo mais rápido no Colab
def animate(i):
    idx = i * passo
    if idx > len(t_total):
        idx = len(t_total)

    # Atualiza ambas as linhas
    linha_total.set_data(t_total[:idx] * 1000, i_total[:idx])
    linha_cc.set_data(t_total[:idx] * 1000, i_dc_total[:idx])

    return linha_total, linha_cc

frames_totais = len(t_total) // passo + 1
ani = animation.FuncAnimation(fig, animate, init_func=init, frames=frames_totais,
                              interval=30, blit=True, repeat=False)

plt.tight_layout()

# Fechamos a figura estática para que o Colab não plote o gráfico vazio junto com o vídeo
plt.close()

# Renderiza a animação como um player HTML interativo direto na saída da célula!
HTML(ani.to_jshtml())

Output hidden; open in https://colab.research.google.com to view.

In [13]:
from google.colab import files

print("Gerando o GIF... aguarde.")
ani.save('curto_circuito.gif', writer='pillow', fps=30)
files.download('curto_circuito.gif')

Gerando o GIF... aguarde.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>